In [ ]:
import pyspark.sql.functions as F


In [ ]:
catalog = dbutils.widgets.get('catalog')

In [ ]:
df = (spark.table(f"{catalog}.silver.spotify_events")
      .filter(F.col("device_type") == "ios")
      .groupBy('Country', 'device_type')
      .agg(F.count('user_id').alias('total_users'),
           F.count('track_id').alias('total_tracks'),
           F.sum(F.col('played_in_mins')).alias('total_played_in_mins'))
).write.mode("overwrite").saveAsTable(f"{catalog}.gold.spotify_ios_events")

In [ ]:
df = (spark.table(f"{catalog}.silver.spotify_events")
      .filter(F.col("device_type") == "android")
      .groupBy('Country', 'device_type')
      .agg(F.count('user_id').alias('total_users'),
           F.count('track_id').alias('total_tracks'),
           F.sum(F.col('played_in_mins')).alias('total_played_in_mins'))
).write.mode("overwrite").saveAsTable(f"{catalog}.gold.spotify_android_events")

In [ ]:
df = (spark.table(f"{catalog}.silver.spotify_events")
      .filter(~F.col("device_type").isin("ios", "android"))
      .groupBy("Country", "device_type")
      .agg(
          F.count("user_id").alias("total_users"),
          F.count("track_id").alias("total_tracks"),
          F.sum("played_in_mins").alias("total_played_in_mins")
      )
)

df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.gold.spotify_other_events")